### MCP Server

You should first run the MCP server:
```
python mcp_servers/weather_server.py
```
---

### Exercise 1: MCP Server for Movie Recommendations with Semantic Caching

Build an MCP server that integrates MongoDB's sample_mflix database with semantic caching for movie descriptions and external API integration for ratings. Your server should implement caching that recognizes when users ask for semantically similar movie queries and returns cached results instead of re-querying the database.

Requirements: Create an MCP server with at least two tools: get_movie_description should first check Redis for a cached response using semantic similarity (cosine similarity > 0.85 on query embeddings), then query MongoDB's sample_mflix.movies collection if no similar cached query exists, format a comprehensive description including plot, cast, and genres, and cache both the query embedding and result for future requests. The get_movie_rating tool should integrate with an external movie API of your choice (TMDB, OMDb, or IMDB API) to fetch current ratings and reviews, caching these results with a shorter TTL (e.g., 24 hours) since ratings can change over time. Implement proper cache key generation using query embeddings, track cache hit/miss rates, and expose Prometheus metrics showing semantic cache effectiveness.


### Exercise 2: User Preference Caching for Weather Units
Extend the existing weather_redis_server.py to automatically remember and apply user temperature unit preferences (Celsius, Fahrenheit, or Kelvin) across sessions, eliminating the need for users to specify their preferred units on every weather query.
Requirements: Modify the get_weather MCP tool to automatically detect when a user explicitly specifies units in their query (e.g., "What's the weather in Paris in Fahrenheit?") and store this preference in Redis with a key pattern like user:{user_id}:pref:units with a 30-day TTL.


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("MONGO_CONNECTION_STRING"):
    print("Connection string for MONGO is not set. Please check your .env file.")
else:
    print("MONGO_CONNECTION_STRING loaded successfully.")

if not os.environ.get("OPENAI_API_KEY"):
    print("API KEY for OPENAI is not set. Please check your .env file.")
else:
    print("OPENAI_API_KEY loaded successfully.")

if not os.environ.get("GROQ_API_KEY"):
    print("API key for Groq is not set. Please check your .env file.")
else:
    print("API key loaded successfully.")

if not os.environ.get("OPENWEATHER_API_KEY"):
    print("API key for OpenWeather is not set. Please check your .env file.")
else:
    print("API key loaded successfully.")

print(os.getenv("MONGO_CONNECTION_STRING"))
print(os.getenv("OPENAI_API_KEY"))
print(os.environ.get("GROQ_API_KEY"))
print(os.environ.get("OPENWEATHER_API_KEY"))

In [ ]:
from langchain_groq import ChatGroq
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import AIMessage

client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "streamable_http",  # HTTP-based remote server
            # Ensure you start your weather server on port 8000
            "url": "http://localhost:8000/mcp",
        }
    }
)

system_message = """You are a helpful assistant with access to weather tools.
    When you receive weather data from tools, always summarize it clearly for the user.
    Never say data is unavailable if the tool returned results."""


tools = await client.get_tools()

model = ChatGroq(
    model="openai/gpt-oss-120b"
)

agent = create_agent(model, tools=tools)

response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What is the weather in London?"},
                  {"role":"system", "content": system_message }
                  ]}
)

aimessages = [message for message in response['messages'] if type(message) == AIMessage]

for message in aimessages:
    print("=" * 20)
    print(message)
